In [4]:
# Install dependencies
# %pip install python-dotenv typesafe_sdk
from dotenv import load_dotenv
import os
load_dotenv()
from typesafe_sdk import TypeSafeClient
MODEL_NAME="jev-latest"

In [ ]:
import re
from dataclasses import dataclass
from decimal import Decimal

from typesafe_sdk import Choice, Noul, TypeSafeClient


@dataclass
class OrderIntent:
    side: str
    instrument: str
    size_fraction: str
    order_type: str
    rate: Decimal | None = None


ALLOWED_INSTRUMENTS = {
    "EURUSD": "EUR/USD",
    "GBPUSD": "GBP/USD",
    "USDJPY": "USD/JPY",
    "EURGBP": "EUR/GBP",
}

SIZE_FRACTIONS = {
    "25_PERCENT": "One quarter of normal/default size",
    "50_PERCENT": "Half of normal/default size",
    "75_PERCENT": "Three quarters of normal/default size",
    "100_PERCENT": "Full normal/default size",
}


RATE_TOKEN_RE = re.compile(
    r"(?<![\w.])(?:\d{1,3}(?:,\d{3})+|\d+)(?:\.\d+)?(?![\w.])"
)
NONE_RATE = "NONE"


def find_numeric_candidates(text: str) -> list[str]:
    """Find numeric spans for Jev to classify; do not guess which one is the FX rate."""
    return list(
        dict.fromkeys(match.group(0) for match in RATE_TOKEN_RE.finditer(text))
    )


def parse_order_instruction(text: str) -> OrderIntent | None:
    rate_candidates = find_numeric_candidates(text)
    rate_criteria = {candidate: None for candidate in rate_candidates}
    rate_criteria[NONE_RATE] = "No explicit FX rate is specified."

    with TypeSafeClient(
    api_key=os.environ["TYPESAFE_API_KEY"],
    base_url=os.environ["TYPESAFE_BASE_URL"]) as client:

        response = client.system_one(
            state={
                "instruction": text,
                "context": {
                    "allowed_instruments": list(ALLOWED_INSTRUMENTS.keys()),
                    "default_order_type": "MARKET",
                    "numeric_candidates": rate_candidates,
                },
            },
            questions={

                # -------------------------
                # Typed arguments
                # -------------------------

                "side": Choice(
                    instructions=(
                        "Determine the trading side explicitly or clearly "
                        "implied by the trader instruction."
                    ),
                    criteria={
                        "BUY": "Trader wants to buy or go long",
                        "SELL": "Trader wants to sell or go short",
                    },
                ),

                "instrument": Choice(
                    instructions=(
                        "Which permitted FX instrument is the trader referring to?"
                    ),
                    criteria={
                        "EURUSD": "EUR/USD, euro-dollar, euro vs dollar",
                        "GBPUSD": "GBP/USD, cable, sterling-dollar",
                        "USDJPY": "USD/JPY, dollar-yen",
                        "EURGBP": "EUR/GBP, euro-sterling",
                    },
                ),

                "size_fraction": Choice(
                    instructions=(
                        "What fraction of the trader's normal/default order "
                        "size is requested?"
                    ),
                    criteria=SIZE_FRACTIONS,
                ),

                "order_type": Choice(
                    instructions="Which supported order type does the trader request?",
                    criteria={
                        "MARKET": "Execute at the current market price",
                        "LIMIT": "Execute only at a specified limit price",
                        "STOP": "Activate when a specified stop level is reached",
                    },
                ),
                "rate": Choice(
                    instructions=(
                        "For a limit or stop order, select the exact numeric candidate "
                        "that is the explicitly stated FX rate. Select NONE if no rate "
                        "is stated. Ignore amounts, sizes, percentages, and other numbers. "
                        "Return only one of the supplied candidates, unchanged."
                    ),
                    criteria=rate_criteria,
                ),

                # -------------------------
                # Ambiguity / safety checks
                # -------------------------

                "side_ambiguous": Noul(
                    instructions=(
                        "The instruction does NOT provide enough information "
                        "to determine BUY versus SELL reliably."
                    )
                ),

                "instrument_ambiguous": Noul(
                    instructions=(
                        "The instruction does NOT identify exactly one permitted "
                        "FX instrument reliably."
                    )
                ),

                "size_ambiguous": Noul(
                    instructions=(
                        "The requested trade size or size fraction is missing "
                        "or ambiguous."
                    )
                ),

                "order_type_ambiguous": Noul(
                    instructions=(
                        "The order type cannot be determined reliably from the "
                        "instruction and known defaults."
                    )
                ),
            },
        )

    side = response.choices["side"]
    instrument = response.choices["instrument"]
    size = response.choices["size_fraction"]
    order_type = response.choices["order_type"]
    rate = response.choices["rate"]

    # Noul gives probability that the proposition is TRUE.
    ambiguities = {
        "side": response.nouls["side_ambiguous"].noul,
        "instrument": response.nouls["instrument_ambiguous"].noul,
        "size": response.nouls["size_ambiguous"].noul,
        "order_type": response.nouls["order_type_ambiguous"].noul,
    }

    rate_value = (
        Decimal(rate.choice.replace(",", ""))
        if rate.choice != NONE_RATE
        else None
    )
    rate_required = order_type.choice in {"LIMIT", "STOP"}

    print("SIDE:", side.choice, side.confidence)
    print("INSTRUMENT:", instrument.choice, instrument.confidence)
    print("SIZE:", size.choice, size.confidence)
    print("ORDER TYPE:", order_type.choice, order_type.confidence)
    print(
        "RATE:",
        rate_value if rate_value is not None else NONE_RATE,
        "confidence:",
        rate.confidence,
    )

    print("AMBIGUITY:", ambiguities)

    # These thresholds are YOUR business rules, not Jev's.
    MIN_CHOICE_CONFIDENCE = 0.85
    MAX_AMBIGUITY = 0.20

    if any(p > MAX_AMBIGUITY for p in ambiguities.values()):
        return None

    if any(
        x.confidence < MIN_CHOICE_CONFIDENCE
        for x in [side, instrument, size, order_type]
    ):
        return None

    if rate_required and rate_value is None:
        print(
            "REJECTED: limit and stop orders require an explicit rate in the instruction."
        )
        return None

    if rate_required and rate.confidence < MIN_CHOICE_CONFIDENCE:
        print("REJECTED: Jev was not confident enough about the specified rate.")
        return None

    return OrderIntent(
        side=side.choice,
        instrument=instrument.choice,
        size_fraction=size.choice,
        order_type=order_type.choice,
        rate=rate_value,
    )




In [ ]:
instructions = [
    "Buy half my usual EUR/USD size at market.",
    "Buy £2m EURUSD at market, split 60% now and rest later.",
    "Sell all my GBP/USD at a limit of 1.3000.",
    "I want to go long on USD/JPY with 75% of my normal size.",
    "Please execute a stop order for EUR/GBP at 0.8500.",
]
if __name__ == "__main__":
    for instruction in instructions:
        print("*************")
        print(f"Instruction: {instruction}")
        order = parse_order_instruction(instruction)
        print(order)